# Human PBMCs

> - Author: Cao Qiang
> - Email: qiangcao2014@gmail.com
> - Created: 2022-7-13
> - Modified: 2024-8-11
---

> - [10k Human PBMCs, 3' v3.1, Chromium X](https://www.10xgenomics.com/cn/datasets/10k-human-pbmcs-3-ht-v3-1-chromium-x-3-1-high)
> - Organism: Homo sapiens
> - Tissue: PBMCs
> - 10X 3' V3.1 scRNA-seq
> - 2021-09-15

---

# Experimental Design

| Sample | Tissue | Full name | Gender | Age | Disease status |
| - | - | - | - | - | - |
| Donor | PBMCs | Peripheral blood mononuclear cells | Female | 25-30 | Healthy |

> - Chromium Single Cell 3' Reagent Kits (v3.1 Chemistry Dual Index, CG000315 Rev C)
> - Chromium X
>     - input: ~ 16,000 cells
>     - recovered: 11,996 cells
> - Illumina NovaSeq 6000
> - Cell Ranger v6.1.0 with --expect-cells = 10000

---

In [ ]:
library(tidyverse)
suppressPackageStartupMessages(library(Seurat))
suppressPackageStartupMessages(library(cowplot))

# 1. Data Import

In [ ]:
data_dir <- "~/data1/single_cell_dataset/dataset_10xGenomics/Hs_PBMCs_3Pv3_10k"

In [ ]:
# dir.create(file.path(data_dir, "result"), 
#            recursive = TRUE, showWarnings = FALSE)
# dir.create(file.path(data_dir, "visualization"), 
#            recursive = TRUE, showWarnings = FALSE)

In [ ]:
scrna <- file.path(
    data_dir, "matrix", "filtered_feature_bc_matrix") |> 
    Read10X() |>
    CreateSeuratObject(project = "PBMCs", min.cells = 3)

In [ ]:
scrna

# 2. Quality Control

## 1). Calculate Metrics

In [ ]:
# Mitochondrial genes
mt <- "(?i)^MT-"
scrna[["percent.mt"]] <- PercentageFeatureSet(scrna, pattern = mt)

In [ ]:
# Ribosomal genes
rp <- "(?i)^(MRP|RP[LS])"
scrna[["percent.rp"]] <- PercentageFeatureSet(scrna, pattern = rp)

In [ ]:
# Hemoglobin genes
hb <- "^(HBA1|HBA2|HBAP1|HBB|HBBP1|HBD|HBE1|HBG1|HBG2|HBM|HBQ1|HBZ|HBZP1)"
scrna[["percent.hb"]] <- PercentageFeatureSet(scrna, pattern = hb)

In [ ]:
head(scrna@meta.data)

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6)
VlnPlot(scrna,
        features = c("nFeature_RNA", "nCount_RNA", "percent.mt", 
                     "percent.rp", "percent.hb"), 
        pt.size = 0.1, ncol = 5) + 
    NoLegend()

In [ ]:
p1 <- FeatureScatter(scrna, feature1 = "nCount_RNA", feature2 = "percent.mt")
p2 <- FeatureScatter(scrna, feature1 = "nCount_RNA", feature2 = "nFeature_RNA")

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6)
plot_grid(p1, p2, ncol = 2)

## Remove doublets

### Filtering low-coverage cells

Remove empty droplets and very low coverage cells to avoid affecting double cell recognition

[DropletUtils](https://github.com/MarioniLab/DropletUtils): Loading from count matrices or molecule information files, identification of cells from empty droplets, removal of barcode-swapped pseudo-cells, and downsampling of the count matrix.

In [ ]:
# Standard Seurat workflow
standard_preprocessing <- function(
    seurat, method = c("lognorm", "sctransform"), 
    ndim = 10, resolution = 0.5) {
    
    method <- match.arg(method)

    if (method == "lognorm") {
        # LogNormalize
        seurat <- NormalizeData(seurat, verbose = FALSE) |>
            FindVariableFeatures(verbose = FALSE) |>
            ScaleData(verbose = FALSE) |>
            RunPCA(verbose = FALSE) |>
            RunUMAP(dims = 1:ndim, verbose = FALSE) |>
            RunTSNE(dims = 1:ndim, verbose = FALSE) |>
            FindNeighbors(dims = 1:ndim, verbose = FALSE) |>
            FindClusters(resolution = resolution, verbose = FALSE)
        message("Completed LogNormalize workflow.")
    } else if (method == "sctransform") {
        # sctransform (very slow)
        seurat <- SCTransform(seurat, verbose = FALSE) |>
            RunPCA(verbose = FALSE) |>
            RunUMAP(dims = 1:ndim, verbose = FALSE) |>
            RunTSNE(dims = 1:ndim, verbose = FALSE) |>
            FindNeighbors(dims = 1:ndim, verbose = FALSE) |>
            FindClusters(resolution = resolution, verbose = FALSE)
        # DefaultAssay(seurat) <- "RNA"
        message("Completed SCTransform workflow.")
    }
    return(seurat)
}

In [ ]:
sc.temp <- standard_preprocessing(scrna)
sc.temp

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 6)
DimPlot(
    sc.temp, reduction = "umap",
    label = TRUE, label.size = 6)

In [ ]:
options(repr.plot.width = 21, repr.plot.height = 6)
FeaturePlot(
    sc.temp, reduction = "umap", ncol = 3,
    features = c("nCount_RNA", "nFeature_RNA", "percent.mt"))

In [ ]:
scrna <- subset(
    scrna, 
    subset = nFeature_RNA > 200 & percent.mt < 20)
scrna

In [ ]:
sc.temp <- standard_preprocessing(scrna)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 6)
DimPlot(
    sc.temp, reduction = "umap",
    label = TRUE, label.size = 6)

In [ ]:
options(repr.plot.width = 21, repr.plot.height = 6)
FeaturePlot(
    sc.temp, reduction = "umap", ncol = 3,
    features = c("nCount_RNA", "nFeature_RNA", "percent.mt"))

### [DoubletFinder](https://github.com/chris-mcginnis-ucsf/DoubletFinder)

1. Generate artificial doublets from existing scRNA-seq data
2. Pre-process merged real-artificial data
3. Perform PCA and use the PC distance matrix to find each cell's proportion of artificial k nearest neighbors (pANN)
4. Rank order and threshold pANN values according to the expected number of doublets

- McGinnis CS, Murrow LM, Gartner ZJ. DoubletFinder: Doublet Detection in Single-Cell RNA Sequencing Data Using Artificial Nearest Neighbors. Cell Syst. 2019 Apr 24;8(4):329-337.e4. doi: 10.1016/j.cels.2019.03.003.

In [ ]:
suppressPackageStartupMessages(library(DoubletFinder))

In [ ]:
# sc.temp <- standard_preprocessing(scrna)
sc.temp

In [ ]:
# Performs pN-pK parameter sweeps on a 10,000-cell subset of
# a pre-processed Seurat object.
set.seed(1234)
sweep.res.list <- paramSweep_v3(sc.temp, PCs = 1:10, sct = FALSE)

In [ ]:
# Summarizes results from doubletFinder::ParamSweep, 
# computing the bimodality coefficient across pN and pK parameter space.

# pK Identification (no ground-truth)
sweep.stats <- summarizeSweep(sweep.res.list, GT = FALSE)

# # pK Identification (ground-truth)
# gt.calls <- temp@meta.data[rownames(sweep.res.list[[1]]), "GT"].
# # When GT = TRUE, the function involves random sampling
# set.seed(1234)
# sweep.stats <- summarizeSweep(sweep.res.list, GT = TRUE, GT.calls = gt.calls)

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 6)
# Computes and visualizes the mean-variance normalized
# bimodality coefficient (BCmvn) score for each pK value
# tested during doubletFinder_ParamSweep.
bcmvn <- find.pK(sweep.stats)

In [ ]:
optimal.pK <- bcmvn$pK[which.max(bcmvn$BCmetric)] |> 
    as.vector() |> as.numeric()
message("optimal pK: ", optimal.pK)

In [ ]:
# Homotypic Doublet Proportion Estimate
# Leverages user-provided cell annotations to model 
# the proportion of homotypic doublets.
homotypic.prop <- modelHomotypic(sc.temp$seurat_clusters)

nCell <- length(Cells(sc.temp))
nExp <- round(nCell / 1000 * 0.008 * nCell)
nExp.adj <- round(nExp * (1 - homotypic.prop))

message("Estimated expected doublets (nExp): ", nExp.adj)

In [ ]:
set.seed(1234)
sc.temp <- doubletFinder_v3(
    sc.temp, PCs = 1:10, pN = 0.25, 
    pK = optimal.pK, nExp = nExp.adj)

In [ ]:
old_col <- paste(c("pANN", "DF.classifications"),
                 0.25, optimal.pK, nExp.adj, sep = "_")
scrna$DoubletFinder.pANN <- sc.temp@meta.data[, old_col[1]]
scrna$DoubletFinder.class <- sc.temp@meta.data[, old_col[2]] |> 
    factor(levels = c("Singlet", "Doublet"))

In [ ]:
table(scrna$DoubletFinder.class)

### [scDblFinder](https://github.com/plger/scDblFinder)

scDblFinder() has two main modes of operation: cluster-based or not. Both perform quite well (see Germain et al., 2021). In general, we recommend the cluster-based approach in datasets with a very clear cluster structure, and the random approach in more complex datasets.

1. Reducing and clustering the data
2. Generating artificial doublets
3. Examining the k-nearest neighbors (kNN) of each cell
4. Training a classifier
5. Thresholding
6. Doublet origins and enrichments

- Germain PL, Lun A, Garcia Meixide C et al. Doublet identification in single-cell sequencing data using scDblFinder [version 2; peer review: 2 approved]. F1000Research 2022, 10:979 (https://doi.org/10.12688/f1000research.73600.2)

In [ ]:
suppressPackageStartupMessages(library(scDblFinder))

In [ ]:
# sce <- standard_preprocessing(scrna) |>
#     as.SingleCellExperiment()
sce <- as.SingleCellExperiment(sc.temp)
sce

In [ ]:
set.seed(1234)
# # random approach in more complex datasets.
# sce <- scDblFinder(sce, verbose = FALSE)
# cluster-based approach in datasets with a very clear cluster structure.
sce <- scDblFinder(sce, clusters = "seurat_clusters", verbose = FALSE)
sce

In [ ]:
# # normal scater-based normalization and PCA
# set.seed(1234)
# sce <- scDblFinder(
#     GetAssayData(sc.temp, slot = "counts"),
#     clusters = Idents(sc.temp), verbose = FALSE)

In [ ]:
scrna$scDblFinder.score <- colData(sce)$scDblFinder.score
scrna$scDblFinder.class <- colData(sce)$scDblFinder.class

In [ ]:
table(scrna$scDblFinder.class)

### Filtering doublet cells

In [ ]:
table(scDblFinder = scrna$scDblFinder.class, 
      DoubletFinder = scrna$DoubletFinder.class)

In [ ]:
sc.temp <- standard_preprocessing(scrna)

In [ ]:
p1 <- VlnPlot(sc.temp, layer = "counts", 
              group.by = "scDblFinder.class", 
              features = c("nCount_RNA", "nFeature_RNA"), 
              pt.size = 0, ncol = 2) +
    NoLegend()
p2 <- VlnPlot(sc.temp, layer = "counts", 
              group.by = "DoubletFinder.class",
              features = c("nCount_RNA", "nFeature_RNA"), 
              pt.size = 0, ncol = 2) +
    NoLegend()

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 6)
plot_grid(p1, p2, ncol = 2)

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6)
DimPlot(
    sc.temp, reduction = "umap", ncol = 2,
    group.by = c("scDblFinder.class", "DoubletFinder.class"))

In [ ]:
Idents(sc.temp) <- apply(
    sc.temp@meta.data[, c("scDblFinder.class", "DoubletFinder.class")],
    1, function(x) paste(x, collapse = "_"))

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 6)
DimPlot(sc.temp, reduction = "umap")

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6)
DimPlot(
    subset(sc.temp, subset = scDblFinder.class == "singlet"),
    reduction = "umap", alpha = 0.1, ncol = 2,
    group.by = c("scDblFinder.class", "DoubletFinder.class"))

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6)
DimPlot(subset(sc.temp, subset = DoubletFinder.class == "Singlet"), 
        reduction = "umap", alpha = 0.1, ncol = 2,
        group.by = c("scDblFinder.class", "DoubletFinder.class"))

In [ ]:
scrna <- subset(
    scrna, 
    subset = DoubletFinder.class == "Singlet" & 
        scDblFinder.class == "singlet")
scrna

## Filtering low-quality cells

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6)
VlnPlot(scrna, layer = "counts", 
        features = c("nFeature_RNA", "nCount_RNA", 
                     "percent.mt", "percent.rp", "percent.hb"), 
        pt.size = 0.1, alpha = 0.1, ncol = 5) + 
    NoLegend()

In [ ]:
options(repr.plot.width = 21, repr.plot.height = 6)
FeaturePlot(
    subset(sc.temp, subset = DoubletFinder.class == "Singlet" & 
           scDblFinder.class == "singlet"), 
    reduction = "umap", ncol = 3,
    features = c("nCount_RNA", "nFeature_RNA", "percent.mt"))

In [ ]:
scrna <- subset(
    scrna, 
    subset = nFeature_RNA > 500 & percent.mt < 20)
scrna

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 6)
VlnPlot(scrna, 
        features = c("nFeature_RNA", "nCount_RNA", "percent.mt",
                     "percent.rp", "percent.hb"), 
        pt.size = 0.1, ncol = 5) + 
    NoLegend()

# 3. Evaluating Noise

## Cell cycle effect

In [ ]:
str(cc.genes)

In [ ]:
scrna <- NormalizeData(scrna, verbose = FALSE) |> 
    CellCycleScoring(s.features = cc.genes$s.genes,
                     g2m.features = cc.genes$g2m.genes)

## Cell dissociation effect

- van den Brink, S., Sage, F., Vértesy, Á. et al. Single-cell sequencing reveals dissociation-induced gene expression in tissue subpopulations. Nat Methods 14, 935–936 (2017). https://doi.org/10.1038/nmeth.4437

In [ ]:
dissociation_genes <- readLines("~/data1/single_cell_dataset/reference_infomation/dissociation_genes.txt")
str(dissociation_genes)

In [ ]:
scrna <- AddModuleScore(
    scrna, name = "Dissociation.Score", 
    features = list(dissociationGenes = toupper(dissociation_genes)))

scrna@meta.data <- scrna@meta.data |> 
    rename("Dis.Score" = "Dissociation.Score1")

## Visualization

In [ ]:
sc.temp <- standard_preprocessing(scrna, resolution = 0.5)

In [ ]:
p1 <- DimPlot(
    sc.temp, reduction = "umap", 
    group.by = c("orig.ident", "Phase", "seurat_clusters"),
    ncol = 3)
p2 <- FeaturePlot(
    sc.temp, reduction = "umap", 
    features = c("percent.mt", "percent.rp", "percent.hb", 
                 "S.Score", "G2M.Score", "Dis.Score"), 
    ncol = 3)

In [ ]:
options(repr.plot.width = 21, repr.plot.height = 18)
plot_grid(p1, p2, ncol = 1, rel_heights = c(1, 2))

In [ ]:
p1 <- VlnPlot(sc.temp, features = c("S.Score", "percent.rp", "Dis.Score"), 
        group.by = "seurat_clusters", pt.size = 0, ncol = 3)
p2 <- DimPlot(sc.temp, reduction = "umap", 
              label = TRUE, label.size = 6)

options(repr.plot.width = 21, repr.plot.height = 6)
plot_grid(p1, p2, ncol = 2, rel_widths = c(2, 1))

In [ ]:
pca <- Embeddings(sc.temp, "pca")[, 1:10]
scores <- sc.temp@meta.data[, c("S.Score", "G2M.Score",
                                "percent.rp", "Dis.Score")]

cors <- apply(pca, 2, function(pc){
    c(cor(pc, scores$S.Score, method = "pearson"),
      cor(pc, scores$G2M.Score, method = "pearson"),
      cor(pc, scores$percent.rp, method = "pearson"),
      cor(pc, scores$Dis.Score, method = "pearson"))
})

rownames(cors) <- c("cor_S", "cor_G2M", "cor_rp", "cor_Dis")
colnames(cors) <- paste0("PC", 1:ncol(cors))

print(cors)

# 4. Normalization

In [ ]:
scrna <- NormalizeData(scrna, verbose = FALSE)

In [ ]:
scrna <- FindVariableGenes(scrna, do.plot = FALSE, verbose = FALSE)

In [ ]:
scrna <- ScaleData(scrna, verbose = FALSE)

# 5. Dimensional Reduction

## PCA

In [ ]:
scrna <- RunPCA(
    scrna, features = VariableFeatures(scrna), npcs = 50)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 6)
DimPlot(scrna, reduction = "pca")

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6)
VizDimLoadings(scrna, dims = 1:2, reduction = "pca", ncol = 2)

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6)
DimHeatmap(scrna, dims = 1:2, cells = 500, balanced = TRUE)

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 6)
ElbowPlot(scrna, ndims = 50)

In [ ]:
# Estimating PCA dimensionality
pca_var <- scrna@reductions$pca@stdev^2
pca_con <- pca_var / sum(pca_var)
pc <- which(cumsum(pca_con) > 0.9 & pca_con < 0.05)[1]
pc

In [ ]:
scrna <- JackStraw(scrna, num.replicate = 100)
scrna

In [ ]:
scrna <- ScoreJackStraw(scrna, dims = 1:20)
scrna

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 6)
JackStrawPlot(scrna, dims = 1:20)

# 6. Integration (Skip)

# 7. Cluster

In [ ]:
nPCs <- 20

In [ ]:
scrna <- RunUMAP(scrna, dims = 1:nPCs, reduction = "pca", 
                 verbose = FALSE)
scrna <- RunTSNE(scrna, dims = 1:nPCs, reduction = "pca", 
                 verbose = FALSE)
scrna

In [ ]:
scrna <- FindNeighbors(scrna, features = VariableFeatures(scrna),
                       reduction = "pca", dims = 1:nPCs,  verbose = FALSE)
scrna

In [ ]:
suppressPackageStartupMessages(library(clustree))

In [ ]:
resolution_clusters <- function (
    seurat, plot_dir, reduction, nPCs = 10, resolution = seq(0.1, 1, 0.1)) {

    for (res in resolution) {
        seurat <- FindClusters(seurat, resolution = res, verbose = FALSE)
        # UMAP
        seurat <- RunUMAP(seurat, dims = 1:nPCs, 
                          reduction = reduction, verbose = FALSE)
        p <- DimPlot(seurat, reduction = "umap", group.by = c("seurat_clusters"), 
                     label = TRUE, label.size = 6, repel = TRUE)
        file.path(plot_dir, paste0("p7.1_UMAP_clusters_", res ,".png")) |>
            ggsave(plot = p, width = 7, height = 6)
        # t-SNE
        seurat <- RunTSNE(seurat, dims = 1:nPCs, 
                          reduction = reduction, verbose = FALSE)
        p <- DimPlot(seurat, reduction = "tsne", group.by = c("seurat_clusters"),
                     label = TRUE, label.size = 6, repel = TRUE)
        file.path(plot_dir, paste0("p7.2_TSNE_clusters_", res ,".png")) |>
            ggsave(plot = p, width = 7, height = 6)
    }
    
    cluster_tree <- clustree(seurat@meta.data, prefix = "RNA_snn_res.", 
                             node_text_size = 6) +
        theme(legend.title = element_text(size = 18), 
              legend.text = element_text(size = 16))
    
    file.path(plot_dir, paste0("p7.3_cluster_tree.png")) |>
        ggsave(plot = cluster_tree, width = 12, height = 10)
    
    return(cluster_tree)
}

In [ ]:
cluster_tree <- resolution_clusters(
    scrna, plot_dir = file.path(data_dir, "visualization"),
    reduction = "pca", nPCs = nPCs, 
    resolution = seq(0.1, 1, 0.1)
)

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 10)
cluster_tree

### Select resolution

In [ ]:
scrna <- FindClusters(scrna, resolution = 0.5, 
                      verbose = FALSE)
scrna <- RunUMAP(scrna, dims = 1:nPCs, reduction = "pca", 
                 verbose = FALSE)
scrna <- RunTSNE(scrna, dims = 1:nPCs, reduction = "pca",
                 verbose = FALSE)
scrna

In [ ]:
p1 <- DimPlot(scrna, reduction = "umap", group.by = "seurat_clusters", 
              label = TRUE, label.size = 6, repel = TRUE)
p2 <- DimPlot(scrna, reduction = "tsne", group.by = "seurat_clusters", 
              label = TRUE, label.size = 6, repel = TRUE)

In [ ]:
options(repr.plot.width = 18, repr.plot.height = 8)
plot_grid(p1, p2, ncol = 2)

In [ ]:
scrna

In [ ]:
# Silhouette Coefficiency
calc_silhouette <- function(
    seurat, reduction = "pca", nPCs = 10, cluster_col = "seurat_clusters") {
    
    clus <- scrna@meta.data[, cluster_col] |> as.integer()
    
    pca_dist <- Embeddings(scrna, reduction)[, 1:nPCs] |> dist()
    sil_mat <- cluster::silhouette(clus, pca_dist)
    mean(sil_mat[, 3])                               
}

In [ ]:
calc_silhouette(scrna, reduction = "PCA", nPCs = nPCs)

In [ ]:
# Modularity
calc_modularity <- function(
    seurat, graph_name = "RNA_snn", cluster_col = "seurat_clusters",
    sample_threshold = 20000, sample_size = 10000) {

    if (ncol(seurat) > sample_threshold) {
        message(sprintf(
            "⚠️ Detected %d cells > %d. Sampling %d cells for modularity calculation...",
            ncol(seurat), sample_threshold, sample_size
        ))
        sampled_cells <- sample(colnames(seurat), sample_size)
        seurat <- subset(seurat, cells = sampled_cells)
    }
    
    clus <- seurat@meta.data[, cluster_col] |> as.integer()
    
    seurat@graphs[[graph_name]] |> as.matrix() |>
        igraph::graph_from_adjacency_matrix(
            mode = "undirected", weighted = TRUE) |>
        igraph::modularity(clus)
}

In [ ]:
calc_modularity(scrna)

In [ ]:
mean_cluster_logfc <- function(
    seurat, min.pct = 0.25, logfc.threshold = 0.25, top_n = 10) {
    
    FindAllMarkers(seurat, logfc.threshold = logfc.threshold, min.pct = min.pct,
                   only.pos = TRUE, verbose = FALSE) |>
        group_by(cluster) |>
        top_n(n = top_n, wt = avg_log2FC) |>
        summarise(med = median(avg_log2FC)) |>
        summarise(mean_median_logFC = mean(med)) |>
        pull(mean_median_logFC)
}

In [ ]:
mean_cluster_logfc(scrna)

# 8. Find marker genes

In [ ]:
time_taken <- system.time({
    sc.markers <- FindAllMarkers(scrna, logfc.threshold = 0.25, min.pct = 0.1,
                                 only.pos = FALSE, verbose = FALSE)
})

cat("Elapsed:", format(.POSIXct(time_taken["elapsed"], tz = "UTC"), "%H:%M:%S"), "\n")

In [ ]:
head(sc.markers)

In [ ]:
p1 <- VlnPlot(scrna, features = "CD14")
p2 <- RidgePlot(scrna, features = "CD14")
p3 <- FeaturePlot(scrna, reduction = "umap", features = "CD14")

In [ ]:
options(repr.plot.width = 21, repr.plot.height = 6)
plot_grid(p1, p2, p3, ncol = 3, rel_widths = c(1.5, 1, 1))

In [ ]:
sc.markers |> group_by(cluster) |> 
    dplyr::filter(avg_log2FC > 1) |> 
    dplyr::filter(p_val < 0.05) |> 
    summarise(length(gene)) |>
    t()

In [ ]:
top10 <- sc.markers |> 
    group_by(cluster) |> 
    dplyr::filter(avg_log2FC > 1) |> 
    dplyr::filter(p_val < 0.05) |> 
    top_n(10, wt = avg_log2FC)

top5 <- sc.markers |> 
    group_by(cluster) |> 
    dplyr::filter(avg_log2FC > 1) |> 
    dplyr::filter(p_val < 0.05) |> 
    top_n(5, wt = avg_log2FC)

## Visualization

In [ ]:
options(repr.plot.width = 18, repr.plot.height = 12)
DoHeatmap(scrna, features = top5$gene, size = 8) +
    # scale_fill_gradientn(colors = PurpleAndYellow()) +
    theme(axis.text = element_text(size = 12),
          legend.title = element_text(size = 18), 
          legend.text = element_text(size = 16))

In [ ]:
options(repr.plot.width = 18, repr.plot.height = 6)
DotPlot(scrna, features = top5$gene, dot.scale = 8) + 
    RotatedAxis() + 
    theme(axis.title = element_text(size = 18),
          axis.text = element_text(size = 12),
          legend.title = element_text(size = 18), 
          legend.text = element_text(size = 16))

In [ ]:
for (index in seq(1:length(top10$gene))) {
    plot_dir <- file.path(
        data_dir, "visualization", "p8.2_cluster_top10_genes",
        paste0("cluster_", top10$cluster[index]))
    
    if (!dir.exists(plot_dir)) {
        dir.create(plot_dir, recursive = TRUE)
    }
    
    p <- FeaturePlot(scrna, reduction = "umap", 
                     features = top10$gene[index])
    file.path(plot_dir, paste0("umap_", top10$gene[index], ".png")) |>
        ggsave(plot = p, width = 7, height = 6)
    
    p <- FeaturePlot(scrna, reduction = "tsne", 
                     features = top10$gene[index])
    file.path(plot_dir, paste0("tsne_", top10$gene[index], ".png")) |>
        ggsave(plot = p, width = 7, height = 6)
}

# 9. Annotation

## Automated

### [ScType](https://github.com/IanevskiAleksandr/sc-type/)

ScType: Fully-automated and ultra-fast cell-type identification using specific marker combinations from single-cell transcriptomic data

- Ianevski, A., Giri, A.K. & Aittokallio, T. Fully-automated and ultra-fast cell-type identification using specific marker combinations from single-cell transcriptomic data. Nat Commun 13, 1246 (2022). https://doi.org/10.1038/s41467-022-28803-w

In [ ]:
source("https://raw.githubusercontent.com/IanevskiAleksandr/sc-type/master/R/sctype_wrapper.R")
db_ <- "https://raw.githubusercontent.com/IanevskiAleksandr/sc-type/master/ScTypeDB_full.xlsx"
db_ <- "https://raw.githubusercontent.com/IanevskiAleksandr/sc-type/master/ScTypeDB_short.xlsx"

sc.temp <- run_sctype(
    scrna, assay = "RNA", scaled = TRUE, known_tissue_type = "Immune system",
    custom_marker_file = db_, name = "sctype_classification")


In [ ]:
suppressPackageStartupMessages(library(openxlsx))
suppressPackageStartupMessages(library(HGNChelper))

In [ ]:
# load gene set preparation function
source("https://raw.githubusercontent.com/IanevskiAleksandr/sc-type/master/R/gene_sets_prepare.R")
# load cell type annotation function
source("https://raw.githubusercontent.com/IanevskiAleksandr/sc-type/master/R/sctype_score_.R")

# get cell-type-specific gene sets from our in-built database (DB)
db_ <- "https://raw.githubusercontent.com/IanevskiAleksandr/sc-type/master/ScTypeDB_full.xlsx"
db_ <- "https://raw.githubusercontent.com/IanevskiAleksandr/sc-type/master/ScTypeDB_short.xlsx"

In [ ]:
# check Seurat object version (scRNA-seq matrix extracted differently in Seurat v4/v5)
seurat_package_v5 <- isFALSE('counts' %in% names(attributes(pbmc[["RNA"]])));
print(sprintf("Seurat object %s is used", ifelse(seurat_package_v5, "v5", "v4")))

# extract scaled scRNA-seq matrix
scRNAseqData_scaled <- if (seurat_package_v5) as.matrix(pbmc[["RNA"]]$scale.data) else as.matrix(pbmc[["RNA"]]@scale.data)

# run ScType
es.max <- sctype_score(scRNAseqData = scRNAseqData_scaled, scaled = TRUE, gs = gs_list$gs_positive, gs2 = gs_list$gs_negative)

# NOTE: scRNAseqData parameter should correspond to your input scRNA-seq matrix. For raw (unscaled) count matrix set scaled = FALSE
# When using Seurat, we use "RNA" slot with 'scale.data' by default. Please change "RNA" to "SCT" for sctransform-normalized data,
# or to "integrated" for joint dataset analysis. To apply sctype with unscaled data, use e.g. pbmc[["RNA"]]$counts or pbmc[["RNA"]]@counts, with scaled set to FALSE.

# merge by cluster
cL_resutls <- do.call("rbind", lapply(unique(pbmc@meta.data$seurat_clusters), function(cl){
    es.max.cl = sort(rowSums(es.max[ ,rownames(pbmc@meta.data[pbmc@meta.data$seurat_clusters==cl, ])]), decreasing = !0)
    head(data.frame(cluster = cl, type = names(es.max.cl), scores = es.max.cl, ncells = sum(pbmc@meta.data$seurat_clusters==cl)), 10)
}))
sctype_scores <- cL_resutls %>% group_by(cluster) %>% top_n(n = 1, wt = scores)  

# set low-confident (low ScType score) clusters to "unknown"
sctype_scores$type[as.numeric(as.character(sctype_scores$scores)) < sctype_scores$ncells/4] <- "Unknown"
print(sctype_scores[,1:3])

In [ ]:
pbmc@meta.data$sctype_classification = ""
for(j in unique(sctype_scores$cluster)){
  cl_type = sctype_scores[sctype_scores$cluster==j,]; 
  pbmc@meta.data$sctype_classification[pbmc@meta.data$seurat_clusters == j] = as.character(cl_type$type[1])
}

DimPlot(pbmc, reduction = "umap", label = TRUE, repel = TRUE, group.by = 'sctype_classification')        


In [ ]:
# load libraries
lapply(c("ggraph","igraph","tidyverse", "data.tree"), library, character.only = T)

# prepare edges
cL_resutls <- cL_resutls[order(cL_resutls$cluster),]; edges = cL_resutls; edges$type = paste0(edges$type,"_",edges$cluster); edges$cluster = paste0("cluster ", edges$cluster); edges = edges[,c("cluster", "type")]; colnames(edges) = c("from", "to"); rownames(edges) <- NULL

# prepare nodes
nodes_lvl1 <- sctype_scores[,c("cluster", "ncells")]; nodes_lvl1$cluster = paste0("cluster ", nodes_lvl1$cluster); nodes_lvl1$Colour = "#f1f1ef"; nodes_lvl1$ord = 1; nodes_lvl1$realname = nodes_lvl1$cluster; nodes_lvl1 = as.data.frame(nodes_lvl1); nodes_lvl2 = c(); 
ccolss <- c("#5f75ae","#92bbb8","#64a841","#e5486e","#de8e06","#eccf5a","#b5aa0f","#e4b680","#7ba39d","#b15928","#ffff99", "#6a3d9a","#cab2d6","#ff7f00","#fdbf6f","#e31a1c","#fb9a99","#33a02c","#b2df8a","#1f78b4","#a6cee3")
for (i in 1:length(unique(cL_resutls$cluster))){
  dt_tmp = cL_resutls[cL_resutls$cluster == unique(cL_resutls$cluster)[i], ]; nodes_lvl2 = rbind(nodes_lvl2, data.frame(cluster = paste0(dt_tmp$type,"_",dt_tmp$cluster), ncells = dt_tmp$scores, Colour = ccolss[i], ord = 2, realname = dt_tmp$type))
}
nodes <- rbind(nodes_lvl1, nodes_lvl2); nodes$ncells[nodes$ncells<1] = 1;
files_db <- openxlsx::read.xlsx(db_)[,c("cellName","shortName")]; files_db = unique(files_db); nodes = merge(nodes, files_db, all.x = T, all.y = F, by.x = "realname", by.y = "cellName", sort = F)
nodes$shortName[is.na(nodes$shortName)] = nodes$realname[is.na(nodes$shortName)]; nodes = nodes[,c("cluster", "ncells", "Colour", "ord", "shortName", "realname")]

mygraph <- graph_from_data_frame(edges, vertices=nodes)

# Make the graph
gggr <- ggraph(mygraph, layout = 'circlepack', weight=I(ncells)) + 
  geom_node_circle(aes(filter=ord==1,fill=I("#F5F5F5"), colour=I("#D3D3D3")), alpha=0.9) + geom_node_circle(aes(filter=ord==2,fill=I(Colour), colour=I("#D3D3D3")), alpha=0.9) +
  theme_void() + geom_node_text(aes(filter=ord==2, label=shortName, colour=I("#ffffff"), fill="white", repel = !1, parse = T, size = I(log(ncells,25)*1.5)))+ geom_node_label(aes(filter=ord==1,  label=shortName, colour=I("#000000"), size = I(3), fill="white", parse = T), repel = !0, segment.linetype="dotted")
  
DimPlot(pbmc, reduction = "umap", label = TRUE, repel = TRUE, cols = ccolss)+ gggr

## Manual

### Marker genes

### Typical genes

In [ ]:
geneset <- c("LYZ", "MS4A1", "NCAM1", "CD3D", "CD8A", "CD4")
p1 <- VlnPlot(
    scrna, features = geneset, ncol = 1, pt.size = 0)
p2 <- RidgePlot(
    scrna, features = geneset, ncol = 2)
p3 <- FeaturePlot(
    scrna, reduction = "umap", features = geneset, ncol = 2)

options(repr.plot.width = 28, repr.plot.height = 18)
plot_grid(p1, p2, p3, ncol = 3, rel_widths = c(1, 1, 2))

In [ ]:
geneset <- c("CD4", "CD8A", "CD8B", "CCR7", "IL7R", 
             "NKG7", "GZMB", "GZMK", "KLRG1", "EOMES", 
             "FOXP3", "KLRB1", "TRDC", "LTB", "CCL5")

options(repr.plot.width = 20, repr.plot.height = 12)
FeaturePlot(scrna, features = geneset, reduction = "umap", 
            cols = c("lightgrey", 'red'), ncol = 5) & 
    NoLegend() & NoAxes() &
    theme(panel.border = element_rect(color = "black", linewidth = 1))

In [ ]:
# Classical monocytes: "CD14", "LYZ", "S100A8", "S100A9", "S100A12"
# Non-classical monocytes: "FCGR3A", "LST1", "IFITM3", "MS4A7", "LILRB1"
# Intermediate monocytes: "CD14", "FCGR3A", "HLA-DRA", "HLA-DPB1", "CTSL", "CTSS", "LILRA1", "CD74"

geneset <- c("CD14", "LYZ", "S100A8", "S100A9", "S100A12",
             "FCGR3A", "LST1", "IFITM3", "MS4A7", "LILRB1",
             "HLA-DRA", "CTSL", "CTSS", "LILRA1", "CD74")

options(repr.plot.width = 20, repr.plot.height = 12)
FeaturePlot(scrna, features = geneset, reduction = "umap", 
            cols = c("lightgrey", 'red'), ncol = 5) & 
    NoLegend() & NoAxes() &
    theme(panel.border = element_rect(color = "black", linewidth = 1))

In [ ]:
# cDC1: "CLEC9A", "XCR1", "IDO1", "C1orf54"
# cDC2: "CD1C", "FCN1", "PILRA"
# pDC: "PTCRA", "SMIM5"
# ASDC: "AXL", "SIGLEC6", "HAMP"

geneset <- c("CST3", "CD74", "BATF3", 
             "CLEC10A", "FCER1A", "TCF4", "IL3RA",
             "CLEC9A", "XCR1", "IDO1", "C1orf54",
             "CD1C", "FCN1", "PTLRA", 
             "PTCRA", "SMIM5",
             "AXL", "SIGLEC6", "HAMP")

options(repr.plot.width = 20, repr.plot.height = 12)
FeaturePlot(scrna, features = geneset, reduction = "umap", 
            cols = c("lightgrey", 'red'), ncol = 5) & 
    NoLegend() & NoAxes() &
    theme(panel.border = element_rect(color = "black", linewidth = 1))

In [ ]:
suppressPackageStartupMessages(library(SeuratDisk))

In [ ]:
output_dir <- file.path(
    data_dir, "result", "s11_annotation_seurat_V5.h5Seurat")
SaveH5Seurat(scrna, filename = output_dir, verbose = FALSE)

In [ ]:
suppressPackageStartupMessages(library("Nebulosa"))

In [ ]:
options(repr.plot.width = 21, repr.plot.height = 6)
plot_density(sc.inte, c("CD4", "CCR7"), joint = TRUE)

In [ ]:
# Seurat降级
remove.packages('Seurat')
pkgs = c( 'mixtools', 'lars', 'dtw', 'doSNOW', 'hdf5r' ) 
BiocManager::install(pkgs,ask = F,update = F)
# 以后只需要修改这个版本号即可
packageurl <- "https://cran.r-project.org/src/contrib/Archive/Seurat/Seurat_2.3.4.tar.gz"
install.packages(packageurl, repos=NULL, type="source")
# 最后检查一下包的版本
library('Seurat')
packageVersion('Seurat')

In [ ]:
library(Seurat)
sce.10x <- Read10X(data.dir = '~/four-PBMC-mtx/SRR7722939/')
sce1 <- CreateSeuratObject(raw.data = sce.10x, min.cells = 60, 
                           min.genes = 200, project = "SRR7722939") 

sce.10x <- Read10X(data.dir = '~/four-PBMC-mtx/SRR7722940/')
sce2 <- CreateSeuratObject(raw.data = sce.10x, min.cells = 60, 
                           min.genes = 200, project = "SRR7722940") 

sce.10x <- Read10X(data.dir = '~/four-PBMC-mtx/SRR7722941/')
sce3 <- CreateSeuratObject(raw.data = sce.10x, min.cells = 60, 
                           min.genes = 200, project = "SRR7722941") 

sce.10x <- Read10X(data.dir = '~/four-PBMC-mtx/SRR7722942/')
sce4 <- CreateSeuratObject(raw.data = sce.10x, min.cells = 60, 
                           min.genes = 200, project = "SRR7722942") 

sce1;sce2;sce3;sce4
## An object of class seurat in project SRR7722939 
##  6163 genes across 2047 samples.
## An object of class seurat in project SRR7722940 
##  4267 genes across 1073 samples.
## An object of class seurat in project SRR7722941 
##  5480 genes across 4311 samples.
## An object of class seurat in project SRR7722942 
##  6427 genes across 4025 samples.


# batch running
folders <- list.files('../../cellranger/four-PBMC-mtx/',pattern = '^SRR')
folders
# [1] "SRR7722939" "SRR7722940" "SRR7722941" "SRR7722942"

library(Seurat)
sceList <- lapply(folders,
                  function(folder){
                      CreateSeuratObject(counts = Read10X(folder),
                                         project = folder ,
                                         min.cells = 60,
                                         min.features  = 200)
                  }
                 )

In [ ]:
# 区分对象名称
sce1@meta.data$group <- "PBMC_Pre"
sce2@meta.data$group <- "PBMC_EarlyD27"
sce3@meta.data$group <- "PBMC_RespD376"
sce4@meta.data$group <- "PBMC_ARD614"

# 区分细胞名
colnames(sce1@data) <- paste0("PBMC_Pre.",colnames(sce1@data))
colnames(sce2@data) <- paste0("PBMC_EarlyD27.",colnames(sce2@data))
colnames(sce3@data) <- paste0("PBMC_RespD376.",colnames(sce3@data))
colnames(sce4@data) <- paste0("PBMC_ARD614.",colnames(sce4@data))

In [ ]:
sce1 <- NormalizeData(sce1)
sce1 <- ScaleData(sce1, display.progress = F) 
sce2 <- NormalizeData(sce2)
sce2 <- ScaleData(sce2, display.progress = F) 
sce3 <- NormalizeData(sce3)
sce3 <- ScaleData(sce3, display.progress = F) 
sce4 <- NormalizeData(sce4)
sce4 <- ScaleData(sce4, display.progress = F) 

In [ ]:
sce1 <- FindVariableGenes(sce1, do.plot = F)
sce2 <- FindVariableGenes(sce2, do.plot = F)
sce3 <- FindVariableGenes(sce3, do.plot = F)
sce4 <- FindVariableGenes(sce4, do.plot = F)
# 前1000个HVGs结果记录下来
g.1 <- head(rownames(sce1@hvg.info), 1000)
g.2 <- head(rownames(sce2@hvg.info), 1000)
g.3 <- head(rownames(sce3@hvg.info), 1000)
g.4 <- head(rownames(sce4@hvg.info), 1000)
# 再取交集
genes.use <- unique(c(g.1, g.2,g.3, g.4))
genes.use <- intersect(genes.use, rownames(sce1@scale.data))
genes.use <- intersect(genes.use, rownames(sce2@scale.data))
genes.use <- intersect(genes.use, rownames(sce3@scale.data))
genes.use <- intersect(genes.use, rownames(sce4@scale.data))
head(genes.use)
## [1] "hg38_S100A9"    "hg38_PPBP"      "hg38_S100A8"    "hg38_MALAT1"   
## [5] "hg38_PF4"       "hg38_HIST1H2AC"

# 最后得到1450个共有高变异基因
length(genes.use)
## [1] 1450

In [ ]:
sce.comb <- RunMultiCCA(list(sce1,sce2,sce3,sce4),
                        add.cell.ids = c("PBMC_Pre.","PBMC_EarlyD27.","PBMC_RespD376.","PBMC_ARD614."),
                        genes.use = genes.use,
                        num.cc = 30)

DimPlot(object = sce.comb,
        reduction.use = "cca",
        group.by = "group",
        pt.size = 0.5,
        do.return = TRUE)

In [ ]:
# 以下代码基于Seurat 3.1
sceList[[1]]@meta.data$orig.ident <- "PBMC_Pre"
sceList[[2]]@meta.data$orig.ident <- "PBMC_EarlyD27"
sceList[[3]]@meta.data$orig.ident <- "PBMC_RespD376"
sceList[[4]]@meta.data$orig.ident <- "PBMC_ARD614"

sce.big <- merge(sceList[[1]], 
                 y = c(sceList[[2]],sceList[[3]],sceList[[4]]), 
                 add.cell.ids = c("PBMC_Pre.","PBMC_EarlyD27.","PBMC_RespD376.","PBMC_ARD614."), 
                 project = "p1-PBMC")

> table(sce.big$orig.ident)             

  PBMC_Pre SRR7722940 SRR7722941 SRR7722942 
      2047       1074       4311       4028 

sce.big <- SCTransform(sce.big, verbose = FALSE)
sce.big <- RunPCA(sce.big, verbose = FALSE)
sce.big <- RunTSNE(sce.big, verbose = FALSE)
DimPlot(object = sce.big, 
        reduction = "tsne",group.by = 'orig.ident')

In [ ]:
sce.big <- FindNeighbors(sce.big, dims = 1:20)
sce.big <- FindClusters(sce.big, resolution = 0.2)
table(sce.big@meta.data$SCT_snn_res.0.2)
DimPlot(object = sce.big, reduction = "tsne",group.by = 'SCT_snn_res.0.2')

table(sce.big$SCT_snn_res.0.2,sce.big$orig.ident)